# LLM Query Layer
Ask questions about the e-commerce data in plain English. An open-source LLM writes SQL, DuckDB runs it, and an evaluation set of questions with known answers measures how often the generated SQL is correct.

## 1. Load the model
Qwen2.5-Coder-7B-Instruct, an open-source model specialized for code, loaded in 4-bit precision to fit on a free Kaggle GPU. No API key or cost is involved.

In [11]:
!pip install -q -U bitsandbytes duckdb

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-Coder-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
    ),
)

def ask_llm(prompt, max_new_tokens=400):
    # Send one prompt to the model and return only its reply
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

# Quick test
print(ask_llm("Write a SQL query that returns the total revenue from a table called sales with a column revenue."))

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Here is a simple SQL query to calculate the total revenue:

```sql
SELECT SUM(revenue) as Total_Revenue FROM sales;
```

This query uses the `SUM` function in SQL which adds up all the values in the 'revenue' column and gives you the total revenue. The `as` keyword is used to rename the output column to 'Total_Revenue'.


## 2. Load the curated tables
Two small tables exported from the main analysis notebook: daily metrics (one row per day) and category-level daily metrics. Days from November 14 to 17 are flagged because purchase tracking was unreliable.

In [12]:
# List everything attached to this notebook
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__notebook__.ipynb
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__output__.json
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__results__.html
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/custom.css
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/nov_clean.parquet
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct.parquet
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct_clean.parquet
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__results___files/__results___61_0.png
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__results___files/__results___65_0.png
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__results___files/__results___68_1.png
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/results/anomaly_hourly.png
/kaggle/input/noteb

In [13]:
import os, duckdb
con = duckdb.connect()
con.sql("SET memory_limit='20GB'")
con.sql("SET threads=4")
con.sql("SET temp_directory='/tmp/duckdb'")

def find_file(name):
    for root, _, files in os.walk('/kaggle/input'):
        if name in files:
            return os.path.join(root, name)
    return None

print(find_file('oct_clean.parquet'))
print(find_file('nov_clean.parquet'))
print(con.sql(f"SELECT COUNT(*) FROM '{find_file('oct_clean.parquet')}'").fetchall())

/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct_clean.parquet
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/nov_clean.parquet
[(42418544,)]


In [15]:
import re
import pandas as pd

# Load the two small tables exported from the main notebook (section 12)
con.sql(f"""CREATE OR REPLACE TABLE daily_metrics AS
            SELECT * REPLACE (CAST(event_date AS DATE) AS event_date)
            FROM '{find_file('daily_metrics.parquet')}'""")
con.sql(f"""CREATE OR REPLACE TABLE category_daily AS
            SELECT * REPLACE (CAST(event_date AS DATE) AS event_date)
            FROM '{find_file('category_daily.parquet')}'""")

print(con.sql("SELECT COUNT(*) FROM daily_metrics").fetchall(),
      con.sql("SELECT COUNT(*) FROM category_daily").fetchall())

[(61,)] [(854,)]


## 3. Text-to-SQL
The model receives a description of the tables and a question, and must return a single read-only SQL query. The query is checked before running: only one SELECT statement is allowed, so the model cannot modify or delete data.

In [16]:
categories = ', '.join(r[0] for r in con.sql("SELECT DISTINCT category FROM category_daily ORDER BY 1").fetchall())

SCHEMA = f"""
You can query two DuckDB tables covering an online store from 2019-10-01 to 2019-11-30.

Table daily_metrics: one row per day.
- event_date (DATE)
- day_of_week (TEXT, e.g. 'Friday')
- events, active_users, views, purchases, buyers (INTEGER counts for that day)
- revenue (DOUBLE, total value of purchased items)
- conversion (DOUBLE, buyers / active_users, between 0 and 1)
- avg_purchase_price (DOUBLE, revenue / purchases)
- items_per_buyer (DOUBLE, purchases / buyers)
- data_quality_flag (TEXT, 'ok' or 'purchase_tracking_unreliable'; Nov 14-17 are unreliable)

Table category_daily: one row per day and top-level product category.
- event_date (DATE)
- category (TEXT, one of: {categories})
- views, carts, purchases (INTEGER counts)
- revenue (DOUBLE)

Black Friday was 2019-11-29.
"""

def build_prompt(question):
    return f"""{SCHEMA}
Write one DuckDB SQL query that answers the question below.
Return only the SQL inside a ```sql code block, with no explanation.

Question: {question}"""

def extract_sql(text):
    # Take the contents of the first code block; fall back to the whole reply
    match = re.search(r"```(?:sql)?\s*(.*?)```", text, re.S | re.I)
    return (match.group(1) if match else text).strip().rstrip(';').strip()

FORBIDDEN = re.compile(r"\b(insert|update|delete|drop|create|alter|attach|copy|pragma|install|load|export)\b", re.I)

def is_safe(sql):
    # Allow a single read-only query only
    starts_ok = sql.lower().startswith(('select', 'with'))
    return starts_ok and ';' not in sql and not FORBIDDEN.search(sql)

def answer(question):
    sql = extract_sql(ask_llm(build_prompt(question)))
    if not is_safe(sql):
        return {'question': question, 'sql': sql, 'result': None, 'error': 'blocked: not a single read-only query'}
    try:
        return {'question': question, 'sql': sql, 'result': con.sql(sql).df(), 'error': None}
    except Exception as e:
        return {'question': question, 'sql': sql, 'result': None, 'error': str(e)}

In [17]:
for q in [
    "What was total revenue on Black Friday?",
    "Which category had the highest revenue in October?",
    "What was the average daily conversion rate in November, excluding days with unreliable purchase tracking?",
]:
    r = answer(q)
    print('Q:', r['question'])
    print('SQL:', r['sql'])
    print('Error:', r['error'])
    print(r['result'], '\n' + '-' * 60)

Q: What was total revenue on Black Friday?
SQL: SELECT SUM(revenue) AS black_friday_revenue
FROM daily_metrics
WHERE event_date = '2019-11-29'
Error: None
   black_friday_revenue
0            9635318.38 
------------------------------------------------------------
Q: Which category had the highest revenue in October?
SQL: SELECT category, SUM(revenue) AS total_revenue
FROM category_daily
WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 1
Error: None
      category  total_revenue
0  electronics   1.764456e+08 
------------------------------------------------------------
Q: What was the average daily conversion rate in November, excluding days with unreliable purchase tracking?
SQL: SELECT AVG(conversion) AS avg_conversion_rate
FROM daily_metrics
WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30'
AND data_quality_flag = 'ok'
Error: None
   avg_conversion_rate
0             0.066754 
--------------------------------------------

## 4. Evaluation set
31 questions across 7 types, each with a gold SQL query that defines the correct answer. The model's query and the gold query are both run, and their results compared. Grouping questions by type shows which kinds of questions the model handles reliably and which it does not.

In [19]:
EVAL_SET = [
    # A. Simple lookups: one value for one day
    ('A_lookup', "How many active users were there on 2019-10-15?",
     "SELECT active_users FROM daily_metrics WHERE event_date = '2019-10-15'"),
    ('A_lookup', "What was the conversion rate on Black Friday?",
     "SELECT conversion FROM daily_metrics WHERE event_date = '2019-11-29'"),
    ('A_lookup', "How many purchases were made on 2019-11-30?",
     "SELECT purchases FROM daily_metrics WHERE event_date = '2019-11-30'"),
    ('A_lookup', "What was the average purchase price on 2019-10-01?",
     "SELECT avg_purchase_price FROM daily_metrics WHERE event_date = '2019-10-01'"),
    ('A_lookup', "Which day of the week was 2019-11-01?",
     "SELECT day_of_week FROM daily_metrics WHERE event_date = '2019-11-01'"),

    # B. Aggregations over a period
    ('B_aggregate', "What was total revenue in October?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('B_aggregate', "How many purchases were made in total in November?",
     "SELECT SUM(purchases) FROM daily_metrics WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30'"),
    ('B_aggregate', "What was the average number of daily active users in October?",
     "SELECT AVG(active_users) FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('B_aggregate', "What was total revenue from November 22 to November 28?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE event_date BETWEEN '2019-11-22' AND '2019-11-28'"),
    ('B_aggregate', "What was the maximum number of daily views in October?",
     "SELECT MAX(views) FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('B_aggregate', "On how many days did revenue exceed 8 million?",
     "SELECT COUNT(*) FROM daily_metrics WHERE revenue > 8000000"),

    # C. Data quality awareness: must use the data_quality_flag
    ('C_quality', "What was total revenue in November, excluding days with unreliable purchase tracking?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30' AND data_quality_flag = 'ok'"),
    ('C_quality', "How many days have unreliable purchase tracking?",
     "SELECT COUNT(*) FROM daily_metrics WHERE data_quality_flag <> 'ok'"),
    ('C_quality', "Which day had the highest revenue, excluding days with unreliable purchase tracking?",
     "SELECT event_date FROM daily_metrics WHERE data_quality_flag = 'ok' ORDER BY revenue DESC LIMIT 1"),
    ('C_quality', "What was the average daily conversion rate in November, excluding days with unreliable purchase tracking?",
     "SELECT AVG(conversion) FROM daily_metrics WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30' AND data_quality_flag = 'ok'"),

    # D. Ranking
    ('D_ranking', "Which day in October had the most active users?",
     "SELECT event_date FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31' ORDER BY active_users DESC LIMIT 1"),
    ('D_ranking', "What are the top 3 categories by total revenue over the whole period?",
     "SELECT category FROM category_daily GROUP BY category ORDER BY SUM(revenue) DESC LIMIT 3"),
    ('D_ranking', "Which category had the most cart events in November?",
     "SELECT category FROM category_daily WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30' GROUP BY category ORDER BY SUM(carts) DESC LIMIT 1"),
    ('D_ranking', "Which day in October had the lowest conversion rate?",
     "SELECT event_date FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31' ORDER BY conversion ASC LIMIT 1"),

    # E. Comparisons and derived metrics
    ('E_compare', "By what percentage was revenue on Black Friday higher than on the previous Friday, November 22? Give the answer as a percentage.",
     """SELECT 100.0 * (MAX(CASE WHEN event_date = '2019-11-29' THEN revenue END)
                     / MAX(CASE WHEN event_date = '2019-11-22' THEN revenue END) - 1)
        FROM daily_metrics"""),
    ('E_compare', "What was total revenue in November minus total revenue in October?",
     """SELECT SUM(CASE WHEN event_date BETWEEN '2019-11-01' AND '2019-11-30' THEN revenue END)
             - SUM(CASE WHEN event_date BETWEEN '2019-10-01' AND '2019-10-31' THEN revenue END)
        FROM daily_metrics"""),
    ('E_compare', "What was the overall conversion rate in October, computed as total buyers divided by total active users?",
     "SELECT SUM(buyers)::DOUBLE / SUM(active_users) FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('E_compare', "What percentage of total revenue on Black Friday came from the electronics category?",
     """SELECT 100.0 * SUM(CASE WHEN category = 'electronics' THEN revenue END) / SUM(revenue)
        FROM category_daily WHERE event_date = '2019-11-29'"""),
    ('E_compare', "How many more purchases were made on Black Friday than on the previous Friday, November 22?",
     """SELECT MAX(CASE WHEN event_date = '2019-11-29' THEN purchases END)
             - MAX(CASE WHEN event_date = '2019-11-22' THEN purchases END)
        FROM daily_metrics"""),

    # F. Date logic
    ('F_dates', "What was the average daily revenue on Fridays in October?",
     "SELECT AVG(revenue) FROM daily_metrics WHERE day_of_week = 'Friday' AND event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('F_dates', "Which day of the week had the highest average conversion rate, excluding days with unreliable purchase tracking?",
     "SELECT day_of_week FROM daily_metrics WHERE data_quality_flag = 'ok' GROUP BY day_of_week ORDER BY AVG(conversion) DESC LIMIT 1"),
    ('F_dates', "What was total revenue in the first 7 days of November?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE event_date BETWEEN '2019-11-01' AND '2019-11-07'"),
    ('F_dates', "What was total revenue on Saturdays and Sundays in October?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE day_of_week IN ('Saturday', 'Sunday') AND event_date BETWEEN '2019-10-01' AND '2019-10-31'"),

    # G. Category questions, including one that needs both tables
    ('G_category', "What was the revenue of the computers category on Black Friday?",
     "SELECT revenue FROM category_daily WHERE category = 'computers' AND event_date = '2019-11-29'"),
    ('G_category', "How many views did the appliances category get in October?",
     "SELECT SUM(views) FROM category_daily WHERE category = 'appliances' AND event_date BETWEEN '2019-10-01' AND '2019-10-31'"),
    ('G_category', "On which day did the electronics category have its highest revenue, excluding days with unreliable purchase tracking?",
     """SELECT c.event_date FROM category_daily c JOIN daily_metrics d USING (event_date)
        WHERE c.category = 'electronics' AND d.data_quality_flag = 'ok'
        ORDER BY c.revenue DESC LIMIT 1"""),
]

print(len(EVAL_SET), 'questions')

31 questions


### Comparing results
Results are compared by value, not by exact SQL text, since many different queries can give the same correct answer. The comparison ignores column names, extra columns, and row order, and allows tiny floating-point differences. For each column in the gold result, the model's result must contain a column with the same values and the same number of rows.

In [20]:
import numpy as np

def _normalize(col):
    # Make dates and text comparable regardless of how DuckDB returned them
    if pd.api.types.is_datetime64_any_dtype(col):
        return col.dt.strftime('%Y-%m-%d')
    return col

def columns_match(a, b):
    a, b = _normalize(a), _normalize(b)
    if len(a) != len(b):
        return False
    if pd.api.types.is_numeric_dtype(a) and pd.api.types.is_numeric_dtype(b):
        x, y = np.sort(a.astype(float).values), np.sort(b.astype(float).values)
        return np.allclose(x, y, rtol=1e-4, atol=1e-6, equal_nan=True)
    return sorted(a.astype(str)) == sorted(b.astype(str))

def results_match(gold, pred):
    if pred is None or len(gold) != len(pred):
        return False
    # Every gold column must be matched by some column in the model's result
    return all(any(columns_match(gold[g], pred[p]) for p in pred.columns) for g in gold.columns)

### Running the evaluation

In [22]:
rows = []
for category, question, gold_sql in EVAL_SET:
    gold = con.sql(gold_sql).df()
    r = answer(question)
    rows.append({
        'type': category,
        'question': question,
        'correct': results_match(gold, r['result']),
        'error': r['error'],
        'model_sql': r['sql'],
        'gold_answer': gold.iloc[0, 0] if len(gold) else None,
    })

eval_results = pd.DataFrame(rows)

print(f"Overall accuracy: {eval_results['correct'].mean():.1%} "
      f"({eval_results['correct'].sum()} of {len(eval_results)})")
eval_results.groupby('type')['correct'].agg(['mean', 'sum', 'count']).round(2)

Overall accuracy: 90.3% (28 of 31)


,mean,sum,count
type,,,
A_lookup,1.00,5,5
B_aggregate,1.00,6,6
C_quality,0.75,3,4
D_ranking,1.00,4,4
E_compare,0.60,3,5
F_dates,1.00,4,4
G_category,1.00,3,3


In [23]:
# Show every question the model got wrong, with its SQL, so failures can be reviewed by hand
pd.set_option('display.max_colwidth', None)
eval_results[~eval_results['correct']][['type', 'question', 'error', 'model_sql']]

,type,question,error,model_sql
13,C_quality,"Which day had the highest revenue, excluding days with unreliable purchase tracking?",None,"SELECT event_date, MAX(revenue) AS max_revenue\nFROM daily_metrics\nWHERE data_quality_flag = 'ok'\nAND event_date BETWEEN '2019-10-01' AND '2019-11-30'\nGROUP BY event_date"
20,E_compare,What was total revenue in November minus total revenue in October?,"Binder Error: Referenced column ""event_date"" was not found because the FROM clause is missing",SELECT \n SUM(CASE WHEN event_date BETWEEN '2019-11-01' AND '2019-11-30' THEN revenue ELSE 0 END) - \n SUM(CASE WHEN event_date BETWEEN '2019-10-01' AND '2019-10-31' THEN revenue ELSE 0 END) AS revenue_difference
23,E_compare,"How many more purchases were made on Black Friday than on the previous Friday, November 22?",None,"SELECT \n (purchases - LAG(purchases) OVER (ORDER BY event_date)) AS purchase_difference\nFROM \n daily_metrics\nWHERE \n event_date IN ('2019-11-22', '2019-11-29')"


### Evaluation results
The model answered 28 of 31 questions correctly (90.3%; 95% confidence interval roughly 75-97%, given the small test set). Lookups, aggregations, ranking, date logic, and category questions were all correct (22 of 22). All three failures involved combining or selecting across rows:
1. "Highest revenue day": used MAX with GROUP BY instead of ORDER BY ... LIMIT 1, returning every day instead of the top one.
2. "November minus October revenue": correct logic, but the FROM clause was missing, so the query failed.
3. "Purchases on Black Friday vs. November 22": used LAG, returning the correct difference plus an extra empty row.

Two of the three failures ran without error and returned plausible-looking results, which is the more dangerous kind of failure because nothing warns the user. These questions were also written with knowledge of the schema, so accuracy on real users' looser phrasing is likely lower.

## 5. Improving the pipeline
Two changes target the failure types found in the evaluation:
1. Prompt rules: a few general SQL guidelines, such as using ORDER BY ... LIMIT 1 to find the top row, and returning one row when the question asks for one number.
2. Retry on error: if a query fails, the error message is sent back to the model, which gets one chance to fix it.

Because the rules were written after looking at the failed questions, testing only on the original 31 would overstate the improvement. A held-out set of 10 new questions, written after the rules and never used to tune them, gives a fairer test.

In [24]:
RULES = """
Follow these rules:
- Always include a FROM clause.
- To find the day or category with the highest or lowest value, use ORDER BY ... DESC or ASC with LIMIT. Do not use MAX or MIN with GROUP BY for this.
- If the question asks for a single number, return exactly one row.
- To compare two specific dates or periods, compute both values in a single row using conditional aggregation, for example SUM(CASE WHEN ... THEN revenue END). Do not use window functions such as LAG for this.
- Use only the tables and columns described above.
"""

def build_prompt_v2(question):
    return f"""{SCHEMA}
{RULES}
Write one DuckDB SQL query that answers the question below.
Return only the SQL inside a ```sql code block, with no explanation.

Question: {question}"""

def run_sql(sql):
    if not is_safe(sql):
        return None, 'blocked: not a single read-only query'
    try:
        return con.sql(sql).df(), None
    except Exception as e:
        return None, str(e)

def answer_v2(question):
    prompt = build_prompt_v2(question)
    sql = extract_sql(ask_llm(prompt))
    result, error = run_sql(sql)
    retried = False

    # If the query failed, show the model its own query and the error, and let it try once more
    if error:
        retried = True
        fix_prompt = f"""{prompt}

Your previous query was:
```sql
{sql}
```
It failed with this error:
{error}

Write a corrected query. Return only the SQL inside a ```sql code block."""
        sql = extract_sql(ask_llm(fix_prompt))
        result, error = run_sql(sql)

    return {'question': question, 'sql': sql, 'result': result, 'error': error, 'retried': retried}

### Held-out questions
Written after the prompt rules, in looser, more user-like wording, and never used to tune the prompt. Several require combining or selecting across rows, the weak spot found in the first evaluation.

In [25]:
HELDOUT_SET = [
    ('H_lookup',  "How much revenue did we make on the Saturday after Black Friday?",
     "SELECT revenue FROM daily_metrics WHERE event_date = '2019-11-30'"),
    ('H_ranking', "Which category sold the most items across the two months?",
     "SELECT category FROM category_daily GROUP BY category ORDER BY SUM(purchases) DESC LIMIT 1"),
    ('H_quality', "What was our best day for conversion in November, not counting the days with tracking problems?",
     """SELECT event_date FROM daily_metrics
        WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30' AND data_quality_flag = 'ok'
        ORDER BY conversion DESC LIMIT 1"""),
    ('H_compare', "How many more active users were there on October 31 than on October 1?",
     """SELECT MAX(CASE WHEN event_date = '2019-10-31' THEN active_users END)
             - MAX(CASE WHEN event_date = '2019-10-01' THEN active_users END)
        FROM daily_metrics"""),
    ('H_compare', "What was the average price of purchased items from November 22 to November 28, computed as total revenue divided by total purchases?",
     "SELECT SUM(revenue) / SUM(purchases) FROM daily_metrics WHERE event_date BETWEEN '2019-11-22' AND '2019-11-28'"),
    ('H_dates',   "On which weekday did people view the most products on average in October?",
     """SELECT day_of_week FROM daily_metrics
        WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31'
        GROUP BY day_of_week ORDER BY AVG(views) DESC LIMIT 1"""),
    ('H_compare', "What percentage of November purchases were in the appliances category?",
     """SELECT 100.0 * SUM(CASE WHEN category = 'appliances' THEN purchases END) / SUM(purchases)
        FROM category_daily WHERE event_date BETWEEN '2019-11-01' AND '2019-11-30'"""),
    ('H_aggregate', "What was the combined revenue of Black Friday and the day after?",
     "SELECT SUM(revenue) FROM daily_metrics WHERE event_date BETWEEN '2019-11-29' AND '2019-11-30'"),
    ('H_aggregate', "How many days in October had more than 220,000 active users?",
     "SELECT COUNT(*) FROM daily_metrics WHERE event_date BETWEEN '2019-10-01' AND '2019-10-31' AND active_users > 220000"),
    ('H_quality', "Which three days had the most purchases, leaving out the days with unreliable tracking?",
     "SELECT event_date FROM daily_metrics WHERE data_quality_flag = 'ok' ORDER BY purchases DESC LIMIT 3"),
]
print(len(HELDOUT_SET), 'held-out questions')

10 held-out questions


### Comparing v1 and v2
v1 is the original pipeline (no rules, no retry). v2 adds the rules and one retry on error. Both are run on the held-out set; v2 is also run on the original 31 questions, and compared with the v1 results from section 4.

In [26]:
def run_eval(answer_fn, eval_set, version, set_name):
    rows = []
    for qtype, question, gold_sql in eval_set:
        gold = con.sql(gold_sql).df()
        r = answer_fn(question)
        rows.append({
            'set': set_name, 'version': version, 'type': qtype, 'question': question,
            'correct': results_match(gold, r['result']),
            'error': r['error'], 'retried': r.get('retried', False), 'model_sql': r['sql'],
        })
    return pd.DataFrame(rows)

# Reuse the v1 results on the original 31 questions from section 4
v1_original = eval_results.assign(set='original', version='v1', retried=False)

v2_original = run_eval(answer_v2, EVAL_SET,    'v2', 'original')
v1_heldout  = run_eval(answer,    HELDOUT_SET, 'v1', 'held-out')
v2_heldout  = run_eval(answer_v2, HELDOUT_SET, 'v2', 'held-out')

all_results = pd.concat([v1_original, v2_original, v1_heldout, v2_heldout], ignore_index=True)

all_results.groupby(['set', 'version'])['correct'].agg(['mean', 'sum', 'count']).round(3)

mean  sum  count
set      version                   
held-out v1       0.600    6     10
         v2       0.700    7     10
original v1       0.903   28     31
         v2       0.935   29     31

In [27]:
# How often did the retry trigger, and did it fix the query?
retries = all_results[all_results['retried']]
print(f"Retries triggered: {len(retries)}, fixed by retry: {retries['correct'].sum()}")

# All remaining failures, for review
all_results[~all_results['correct']][['set', 'version', 'type', 'question', 'error', 'model_sql']]

Retries triggered: 0, fixed by retry: 0


,set,version,type,question,error,model_sql
13,original,v1,C_quality,"Which day had the highest revenue, excluding days with unreliable purchase tracking?",None,"SELECT event_date, MAX(revenue) AS max_revenue\nFROM daily_metrics\nWHERE data_quality_flag = 'ok'\nAND event_date BETWEEN '2019-10-01' AND '2019-11-30'\nGROUP BY event_date"
20,original,v1,E_compare,What was total revenue in November minus total revenue in October?,"Binder Error: Referenced column ""event_date"" was not found because the FROM clause is missing",SELECT \n SUM(CASE WHEN event_date BETWEEN '2019-11-01' AND '2019-11-30' THEN revenue ELSE 0 END) - \n SUM(CASE WHEN event_date BETWEEN '2019-10-01' AND '2019-10-31' THEN revenue ELSE 0 END) AS revenue_difference
23,original,v1,E_compare,"How many more purchases were made on Black Friday than on the previous Friday, November 22?",None,"SELECT \n (purchases - LAG(purchases) OVER (ORDER BY event_date)) AS purchase_difference\nFROM \n daily_metrics\nWHERE \n event_date IN ('2019-11-22', '2019-11-29')"
41,original,v2,B_aggregate,On how many days did revenue exceed 8 million?,None,SELECT COUNT(event_date) AS days_with_revenue_exceeding_8m\nFROM daily_metrics\nWHERE revenue > 8
52,original,v2,E_compare,"What was the overall conversion rate in October, computed as total buyers divided by total active users?",None,"SELECT \n SUM(CASE WHEN event_date BETWEEN '2019-10-01' AND '2019-10-31' THEN buyers ELSE 0 END) AS total_buyers,\n SUM(CASE WHEN event_date BETWEEN '2019-10-01' AND '2019-10-31' THEN active_users ELSE 0 END) AS total_active_users\nFROM \n daily_metrics"
64,held-out,v1,H_quality,"What was our best day for conversion in November, not counting the days with tracking problems?",None,"SELECT event_date, MAX(conversion) AS max_conversion\nFROM daily_metrics\nWHERE event_date BETWEEN '2019-11-01' AND '2019-11-30'\nAND data_quality_flag = 'ok'\nGROUP BY event_date"
65,held-out,v1,H_compare,How many more active users were there on October 31 than on October 1?,None,SELECT (active_users - LAG(active_users) OVER ()) AS active_users_difference\nFROM daily_metrics\nWHERE event_date = '2019-10-31'
66,held-out,v1,H_compare,"What was the average price of purchased items from November 22 to November 28, computed as total revenue divided by total purchases?",None,SELECT AVG(revenue / purchases) AS avg_purchase_price\nFROM daily_metrics\nWHERE event_date BETWEEN '2019-11-22' AND '2019-11-28'
68,held-out,v1,H_compare,What percentage of November purchases were in the appliances category?,"Binder Error: Referenced table ""c"" not found!\nCandidate tables: ""cd""",SELECT \n (SUM(CASE WHEN c.category = 'appliances' THEN cd.purchases ELSE 0 END) * 100.0 / SUM(cd.purchases)) AS percentage_appliances_purchases\nFROM \n category_daily cd\nJOIN \n daily_metrics d ON cd.event_date = d.event_date\nWHERE \n d.event_date BETWEEN '2019-11-01' AND '2019-11-30'
75,held-out,v2,H_compare,How many more active users were there on October 31 than on October 1?,None,SELECT \n (active_users - LAG(active_users) OVER ()) AS active_users_difference\nFROM \n daily_metrics\nWHERE \n event_date = '2019-10-31'


### Improvement results
| Set | v1 (original) | v2 (rules + retry) |
|---|---|---|
| Original 31 questions | 90.3% (28) | 93.5% (29) |
| Held-out 10 questions | 60% (6) | 70% (7) |

1. Wording matters: accuracy dropped from 90% on the original questions to 60% on the held-out questions, which use looser, more user-like phrasing. With only 10 held-out questions the estimate is imprecise (95% interval roughly 31-83%), but the gap is large enough to show that the original set, written with the schema's exact terms, flattered the model.
2. Prompt rules fixed the targeted failures but caused new ones: v2 fixed all three original failures, but broke two questions v1 had answered correctly. It read "8 million" as 8, and returned total buyers and total active users without dividing them, likely over-applying the conditional aggregation rule. Like code changes, prompt changes need a full re-test, not just a re-test of the failed cases.
3. The retry never triggered: none of v2's queries raised an error. All 5 remaining v2 failures were silent wrong answers, so retrying on errors does not address the main risk.
4. Remaining failure types: ignoring an explicitly stated formula (average of daily ratios instead of total revenue divided by total purchases), not following rules (still using LAG after being told not to), using the wrong column (revenue instead of purchases), and misreading units.

Implication: answers from this model should not be trusted without review. A practical deployment would show the generated SQL alongside the result, add automatic checks (for example, one row when one number is asked for, and plausible magnitudes), and use a stronger model or worked examples in the prompt rather than abstract rules.